## Parametrization and Channel Generation

In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from py_newton_optimizer import NewtonOptimizerOptions

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [ ]:
input_mesh = mesh.Mesh("../../../../examples/saddle_low.obj")
# input_mesh = mesh.Mesh("../../../../examples/lilium.msh")

In [ ]:
lines = np.array([[-1.        , -1.        ,  2.415     ],
       [-2.30769231, -1.        ,  3.81923077],
       [ 0.77304965,  1.        , -2.27304965],
       [ 1.29357798,  1.        , -2.94036697],
       [-0.43333333, -1.        ,  1.655     ]])

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(input_mesh, parametrization.lscm(input_mesh))

for i in range(1000): lg.runIteration()

# lg.alphaMin = 1.354641650129586
# lg.alphaMax = 1.3936682623611498

# lg.betaMin = 1.1282764273066557
# lg.betaMax = 1.141976581731514

# lg.alphaMin = 1.2349791815955107 #1.354641650129586
# lg.alphaMax = 1.2827966225656238 #1.3936682623611498

# lg.betaMin = 1.2349791815955107 #1.1282764273066557
# lg.betaMax = 1.2827966225656238 #1.141976581731514


lg.setLines(lines)
lg.alphaMin = 1.0
lg.alphaMax = 1.5

lg.betaMin = 1.0
lg.betaMax = 1.5


(1.3936682623611498, 1.354641650129586, 1.141976581731514, 1.1282764273066557)

print(lg.energy())
lg.runIteration()
print(lg.energy())

for i in range(5000): lg.runIteration()
print(lg.energy())

In [ ]:
visualization.visualize_both(lg)

In [ ]:
lg.energy()

In [ ]:
rparam = parametrization.RegularizedGenericParametrizer(lg)
PET = parametrization.RegularizedGenericParametrizer.EnergyType
list(map(rparam.energy, [PET.Fitting, PET.StretchRegularization, PET.PhiRegularization, PET.DiffRegularization, PET.BendingRegularization]))

### Get Splines

In [ ]:
grid_data = np.load("../../Visualization/grid_data.npy")
grid_pattern_1 = np.load("../../Visualization/grid_pattern_1.npy")
grid_pattern_2 = np.load("../../Visualization/grid_pattern_2.npy")

In [ ]:
augmented_x_scale_factors = np.load("../../Visualization/augmented_x_scale_factors.npy")
augmented_y_scale_factors = np.load("../../Visualization/augmented_y_scale_factors.npy")
augmented_pattern_parameters = np.load("../../Visualization/augmented_pattern_parameters.npy")

In [ ]:
import parametrization_helper

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
splines = parametrization_helper.get_mat_params_over_pattern_params_grid_interpolation(grid_pattern_1, grid_pattern_2, grid_data)

In [ ]:
default_pattern_params = [0.5 * (np.max(grid_pattern_1) + np.min(grid_pattern_1))] * len(lg.getAlphas()) + [0.5 * (np.max(grid_pattern_2) + np.min(grid_pattern_2))] * len(lg.getAlphas())

In [ ]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params)
rparam.patternParamBounds = np.array([[0.5, 2.4], [0, 90]])

In [ ]:
visualization.visualize_both(rparam, height = 4, showBarriers=True)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.RGP, PET.Bending]))

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW, bendRegW = 0.0):
    param.patternRegW = alphaRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    param.diffRegW = 0.0
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = 100
    opts.gradTol = 1e-9
    opts.factorizer = opts.factorizer.CatamariNesdis
    benchmark.reset()
    
    # uv_fixedvars = range(param.phiOffset())
    # uv_fixedvars = range(param.stretchOffset())
    print(param.energy())
    # cr = parametrization.pattern_parametrization_knitro(param, opts.niter, uv_fixedvars)


    cr = parametrization.pattern_parametrization_knitro(param, opts.niter, [param.uOffset(), param.vOffset(), param.phiOffset()])
    benchmark.report()
    return cr

In [ ]:
rparam.bendRegW = 1e-5

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))

In [ ]:
with suppress_stdout(): report = optimize_rparam(rparam, 1e-5, 1e-6, bendRegW = 1e-2)#, 1e-5)

In [ ]:
with suppress_stdout(): report = optimize_rparam(rparam, 1e-5, 1e-6, bendRegW = 1e-5)#, 1e-5)

In [ ]:
rparam.alphaMin, rparam.alphaMax, rparam.getAlphas().min(), rparam.getAlphas().max()

In [ ]:
rparam.betaMin, rparam.betaMax, rparam.getBetas().min(), rparam.getBetas().max()

In [ ]:
visualization.visualize_both(lg, height = 4)

In [ ]:
importlib.reload(visualization)
# visualization.visualize_both(rparam, height = 4, showBarriers=True)
visualization.visualize_pattern(rparam, height = 4, showBarriers=False)

In [ ]:
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
importlib.reload(visualization)

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False)

In [ ]:
# visualization.singularValueHistogramBoth(rparam)

## Upsampling and channel generation

In [ ]:
nsubdiv=4
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)

upsampledStretches

len(upsampledStretches)

len(upsampledAngles)

upsampledStretches = upsampledStretches.reshape(int(len(upsampledStretches)/2), 2, order = 'F')

augmented_pattern_parameters = np.load("../../Visualization/augmented_pattern_parameters.npy")
augmented_x_scale_factors = np.load("../../Visualization/augmented_x_scale_factors.npy")
augmented_y_scale_factors = np.load("../../Visualization/augmented_y_scale_factors.npy")

from scipy.interpolate import griddata

In [ ]:
points = np.concatenate((augmented_x_scale_factors.reshape(-1, 1), augmented_y_scale_factors.reshape(-1, 1)), axis = 1)

In [ ]:
radius_data = griddata(points, augmented_pattern_parameters[:, 0], (upsampledStretches[:, 0], upsampledStretches[:, 1]), method='cubic')

radius_data = radius_data / 2.5 * (np.pi / 2)

angles_data = griddata(points, augmented_pattern_parameters[:, 1], (upsampledStretches[:, 0], upsampledStretches[:, 1]), method='cubic')

angles_data = angles_data / 180 * np.pi

angles_data

In [ ]:
np.sum(np.isnan(radius_data)) + np.sum(np.isnan(angles_data))

In [ ]:
radius_data

(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_cross_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles, radius_data, angles_data, frequency=1)


# pickle.dump((sdfVertices, sdfTris, sdf), open('stripe_sdf_ns4_f100.pkl', 'wb'))

# import pickle, mesh, wall_generation, visualization, numpy as np
# (sdfVertices, sdfTris, sdf) = pickle.load(open('stripe_sdf_ns4_f100.pkl', 'rb'))

importlib.reload(visualization)

import matplotlib as mpl


visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 5, height=5)

In [ ]:
importlib.reload(visualization)
visualization.scalarFieldPlotZeroContourFast(sdfVertices, sdfTris, sdf, width = 15, height=10, cmap = mpl.colormaps["PiYG"])

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=0.1,
                                              minContourLen=0.1)

visualization.plot_line_segments(pts, edges, width=15, height=15)

## Meshing and inflation simulation

In [ ]:
import sheet_meshing, inflation


In [ ]:
m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, pts, edges, triArea=0.1)


In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(iwv) == 1)[0], width=10, height=10)


In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, np.array(iwv) != 0)

In [ ]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), input_mesh.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), input_mesh.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [ ]:
isheet.pressure = 1e-5

In [ ]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

In [ ]:
isheet.pressure = 1e-1

In [ ]:
opts.niter = 2000

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()